# jaxfne Suite No. 2: Spectrolaminar Motif

This tutorial demonstrates the jaxfne Suite No. 2 pipeline. We declare a multi-column layout (V1 and PFC), configure laminar details, run a spontaneous activity simulation, and visualize the 3-panel spectrolaminar profile.

## 1. Learning Objectives

After completing this tutorial, you will understand:
1. **Chainable configuration facade** — how to declare model anatomy, runtime dynamics, cell types, connectivity, and readout modalities using an immutable verb-based Configuration API.
2. **Multi-column layouts** — how to declare separate V1 and PFC cortical column populations and define feedforward and feedback connections between them.
3. **Multimodal readouts** — how to sample simulated source dynamics using various proxy sensors, including MUA-proxy, LFP-proxy, CSD-proxy, EEG-proxy, MEG-proxy, and EMM-proxy.
4. **Spectrolaminar power profiling** — how to extract frequency-depth profiles from laminar extracellular potential signals using JAX-native vis tools.
5. **Optimization search** — how to perform CPU-safe parameter tuning against evocation or spectral objectives.

## 2. Biological/Computational Question

Can separate sensory (V1) and associative (PFC) cortical columns, connected in a feedforward-feedback loop, generate stable laminar and spectral patterns that map onto distinct proxy-level biophysical sensors?

We address this question by simulating a multi-column model and monitoring how spontaneous activity reflects laminar-depth spectrolaminar patterns across superficial and deep layers.

## 3. Mathematical Glossary Flow

Here, we outline the foundational operators and equations defining the emitters, projections, and vis targets:

### 1. Izhikevich Emitter
* **Formal equation:**
  $$\frac{dv}{dt} = 0.04v^2 + 5v + 140 - u + I_{\text{native}}$$
  $$\frac{du}{dt} = a(bv - u)$$
* **Definition of terms:**
  * $v$: Membrane potential (mV).
  * $u$: Membrane recovery variable.
  * $a, b$: Time scale and sensitivity parameters of the recovery variable.
  * $I_{\text{native}}$: Native current drive representing default background inputs.
* **Worded equation:**
  The change in membrane potential over time is the sum of a quadratic voltage activation term, linear scale, offset constant, recovery feedback, and internal driving currents. The change in the recovery variable is scaled by the difference between scaled potential and the recovery variable itself.
* **Implementation location:** [emitters.py](file:///Users/hamednejat/workspace/main/jaxfne/jaxfne/emitters.py)
* **Claim boundary:** Reduced emitter dynamics (phenomenological spiking model), not full conductance-based biophysical reconstruction.

### 2. Source/Readout Bridge
* **Formal equation:**
  $$Y_c(t) = P_c[\text{signals}, \text{source\_proxy}, \text{field\_proxy}](t)$$
* **Definition of terms:**
  * $Y_c(t)$: Multimodal proxy readout at contact or sensor channel $c$ at time $t$.
  * $P_c$: Readout projection operator mapping raw source states to proxy readouts.
  * $\text{signals}$: Active cell state arrays (voltage, spikes).
  * $\text{source\_proxy}, \text{field\_proxy}$: Laminar metadata templates.
* **Worded equation:**
  The multimodal proxy readout is the evaluation of a declarative projection operator mapping raw vectorized emitter variables onto a spatial-depth profile of contacts without solving physical Maxwell PDEs.
* **Implementation location:** [fields.py](file:///Users/hamednejat/workspace/main/jaxfne/jaxfne/fields.py)
* **Claim boundary:** Proxy readout operator representing a computational mapping scaffold, not calibrated physical sensor measurement.

### 3. Spectrolaminar Motif Target
* **Formal equation:**
  $$\text{relative alpha/beta: deeper-biased profile}$$
  $$\text{relative gamma: superficial-biased profile}$$
  $$\text{crossing: near L4 reference depth}$$
* **Definition of terms:**
  * $\text{relative alpha/beta}$: Power distribution focused in deep layers (L5, L6).
  * $\text{relative gamma}$: Power distribution focused in superficial layers (L2/3).
  * $\text{crossing}$: Point of inversion near layer L4.
* **Worded equation:**
  The spectral-depth distribution exhibits a superficial bias for gamma band power, a deep bias for alpha/beta band power, and a reference inversion near mid-column layers.
* **Implementation location:** [vis.py](file:///Users/hamednejat/workspace/main/jaxfne/jaxfne/vis.py)
* **Claim boundary:** Tutorial motif visualization for structural scaffolds, not empirical biological validation.

## 4. Setup and imports

In [ ]:
# Colab-compatible install cell
# !pip install jaxfne

import jaxfne as jtfne
import matplotlib.pyplot as plt
import numpy as np

## 5. Declare Chainable Configuration (Compact Grammar)

In [ ]:
cfg = jtfne.Configuration()
cfg = cfg.runtime(seed=7, dtype="float32", duration_ms=1000.0, dt_ms=0.1)
cfg = cfg.column("V1", layers=["L2/3", "L4", "L5", "L6"], n=80)
cfg = cfg.column("PFC", layers=["L2/3", "L5", "L6"], n=80)
cfg = cfg.cell_types({"E": 0.75, "PV": 0.12, "SST": 0.08, "VIP": 0.05})
cfg = cfg.connectivity(feedforward=("V1", "PFC"), feedback=("PFC", "V1"))
cfg = cfg.set_emitter("izhikevich", "cortical_eig")
cfg = cfg.probes(["MUA-proxy", "LFP-proxy", "CSD-proxy", "EEG-proxy", "MEG-proxy", "EMM-proxy"])

## 6. Claim-Gate Assertion Block

In [ ]:
assert cfg.metadata["truth_mode"] == "truth_safe_unverified"
assert cfg.metadata["physical_amplitude_claim_allowed"] is False
assert cfg.metadata["geometry_mode"] == "declared_metadata_not_solved_3d_pde_grid"
assert cfg.metadata["connectivity_status"] == "declared_metadata_proxy"
assert cfg.metadata["claim_level"] == "computational_scaffold"

print("PASS: all claim gates verified: computational scaffold, proxy-readout, unverified.")

## 7. Construct V1/PFC model

In [ ]:
model = jtfne.construct(cfg)

## 8. Run the simulation

In [ ]:
signals = jtfne.simulate(model, duration_ms=1000.0, dt_ms=0.1, seed=7)

## 9. Spectrolaminar Motif Visualization

In [ ]:
fig = jtfne.vis.spectrolaminar(signals)
plt.show()

## 10. Figures

The generated assets represent the standard visual bundle:
* **Figure 01: V1/PFC 3D Layout** — Column spacing coordinate declaration.
* **Figure 02: Baseline Raster** — Spontaneous spiking profile.
* **Figure 03: Population Rates** — Depth-resolved firing rates.
* **Figure 04: Voltage Traces** — Excitatory and inhibitory membrane potential.
* **Figure 05: MUA-Proxy** — Smoothed multi-unit activity approximation.
* **Figure 06: LFP-Proxy** — Extracellular laminar potentials.
* **Figure 07: CSD-Proxy** — Depth-resolved source/sink transitions.
* **Figure 08: EEG/MEG/EMM Proxy** — Scalp and macro metabolic proxies.
* **Figure 09: Spectrolaminar Heatmap** — 3-panel visual output.
* **Figure 10: Layer-Band Profiles** — Power by cortical depth.
* **Figure 11: Tuning Loss** — Optimization curve.
* **Figure 12: Pre/Post Spectrolaminar** — Spectral profile comparisons.
* **Figure 13: Parameter Trajectory** — Dynamic parameter search path.

## 11. Interpretation

The multi-column spontaneous activity simulation demonstrates how structural feedforward/feedback loops generate localized oscillatory rhythms. By projecting these active patterns through the laminar proxy operator, we recover a spectrolaminar profile where high-frequency (gamma) power peaks in superficial layers (L2/3) and lower-frequency (alpha/beta) power is prominent in deep layers (L5/L6), capturing a classical spectrolaminar motif entirely within a phenomenological computational scaffold.

## 12. Failure Modes

1. **Emitter Saturation:** Re-tuning connection weights too high causes network-wide depolarization block (runaway excitation).
2. **Frequency Locking:** Weak feedback weights can decouple V1 and PFC, leading to isolated, single-frequency oscillations that fail to display the bi-band crossing motif.
3. **Contact Misalignment:** Choosing an arbitrary layout geometry metadata (e.g. negative dz step sizes) shifts the crossing point away from Layer 4.

## 13. Exercises

1. **Tuning Feedback:** Adjust the PFC feedback weight in `cfg.connectivity` and plot how the deep alpha/beta power changes.
2. **Modified Emitter Presets:** Swap the `cortical_eig` emitter preset for an custom drive array to examine baseline firing rates.
3. **Layer Expansion:** Add a `L1` layer to both V1 and PFC, and examine the MUA-proxy readout at the upper contact boundary.

## 14. What this tutorial does NOT claim

* **No biophysical calibration:** The proxy readouts (LFP-proxy, CSD-proxy, etc.) represent arbitrary numerical matrices; they are not calibrated to physical microvolts or microamperes.
* **No biological mechanism proof:** The generated oscillations are products of a simplified, phenomenological Izhikevich system and do not prove any specific biological mechanism.
* **No PDE field solve:** No physical Maxwell equations are solved in 3D grid layouts; readouts are projection proxies based on laminar depth metadata.